# Experiment 1: Core Order-Sensitivity (D3 + D4) — Raw Marginals

**No shuffled baseline** — just raw perplexity marginals for intact, D3, and D4.
101 forward passes per target per condition instead of ~1100.

D3 (swap halves) should show a jump at M=50.
D4 (full reverse) should produce a curve that mirrors the reversed intact curve.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time
from pathlib import Path
from scipy import stats
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

# ========================================================
PILOT_MODE = True    # True = 2 corpora + Llama only
# ========================================================

BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1_disruption_raw')
BASE.mkdir(parents=True, exist_ok=True)

DATA = Path('/content/drive/MyDrive/LRTIA/Data')

ALL_CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'wiki_ja': DATA / 'wiki_multilingual/ja_articles.jsonl',
    'wiki_ko': DATA / 'wiki_multilingual/ko_articles.jsonl',
    'wiki_tr': DATA / 'wiki_multilingual/tr_articles.jsonl',
    'wiki_ar': DATA / 'wiki_multilingual/ar_articles.jsonl',
    'wiki_fi': DATA / 'wiki_multilingual/fi_articles.jsonl',
    'buckeye': DATA / 'buckeye_processed/speaker_concatenated.jsonl',
    'french':  DATA / 'french_oral_processed/per_story.jsonl',
}

if PILOT_MODE:
    CORPORA = {k: v for k, v in ALL_CORPORA.items() if k in ('wiki_zh', 'buckeye')}
    PROBE_LIST = ['llama']
else:
    CORPORA = ALL_CORPORA
    PROBE_LIST = ['llama', 'mistral']

PROBES = {
    'llama': 'unsloth/Meta-Llama-3.1-8B',
    'mistral': 'mistralai/Mistral-7B-v0.1',
}

C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10
M = 50

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'MODE: {"PILOT" if PILOT_MODE else "FULL"} — RAW MARGINALS (no shuffles)')
print(f'Corpora: {list(CORPORA.keys())}')
print(f'Probes: {PROBE_LIST}')

# ~10s per target based on smoke test (101 passes × ~100ms)
est_targets = 86 * 3 if PILOT_MODE else 500 * 3  # targets × conditions
est_hours = est_targets * 10 / 3600
print(f'Est. total: ~{est_hours:.1f} hrs')
print('Setup done')

In [ ]:
# === Disruption functions ===

def d0_no_op(ctx, M=50):
    near = ctx[-M:]
    far = ctx[:-M]
    return far + near

def d3_swap_halves(ctx, M=50):
    near = ctx[-M:]
    far = ctx[:-M]
    return near + far

def d4_full_reverse(ctx):
    return list(reversed(ctx))

# === Perplexity (matches existing pipeline exactly) ===

@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')

def compute_raw_marginals(target_ids, disrupted_ctx):
    """Raw (uncorrected) marginals — no shuffled baseline.
    Just: how does PPL change as we reveal each additional token?"""
    ctx_len = len(disrupted_ctx)
    ppls = []
    
    for c in range(ctx_len + 1):
        if c == 0:
            chunk = list(target_ids)
            ppl = compute_ppl(chunk, 0, len(chunk))
        else:
            prefix = disrupted_ctx[-c:]
            chunk = prefix + list(target_ids)
            ppl = compute_ppl(chunk, len(prefix), len(chunk))
        ppls.append(ppl)
    
    distances = list(range(1, ctx_len + 1))
    marginals = [ppls[d-1] - ppls[d] for d in distances]  # PPL drop from adding token d
    
    return {
        'distances': distances,
        'ppls': ppls,        # PPL at each context length c=0..C
        'marginals': marginals,  # marginal benefit of each token
    }

def process_one_target(full_ids, target_start, target_end, condition_fn, condition_kwargs=None):
    """Process one target region under one disruption condition."""
    target_ids = full_ids[target_start:target_end]
    context = list(full_ids[max(0, target_start - C):target_start])
    if len(context) < C:
        return None
    
    if condition_kwargs:
        disrupted = condition_fn(context, **condition_kwargs)
    else:
        disrupted = condition_fn(context)
    
    return compute_raw_marginals(target_ids, disrupted)

print('Functions defined — NO SHUFFLES, ~10s per target')

In [ ]:
# === D0 Smoke Test + Load Model ===

print('='*60)
print('D0 SMOKE TEST (2 docs, 1 target each, no shuffles)')
print('='*60)

tokenizer = AutoTokenizer.from_pretrained(PROBES['llama'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    PROBES['llama'],
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Model loaded')

corpus = []
with open(CORPORA['wiki_zh']) as f:
    for i, line in enumerate(f):
        if i >= 2: break
        corpus.append(json.loads(line))

t0 = time.time()
for doc in corpus:
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    target_start = int(n * 0.5)
    target_end = min(target_start + TARGET_LEN, n)
    if target_start < MIN_BEFORE:
        continue
    
    print(f'  {doc["doc_id"][:30]}: {n} tokens')
    
    intact = process_one_target(full_ids, target_start, target_end, lambda ctx: ctx)
    d0 = process_one_target(full_ids, target_start, target_end, d0_no_op, {'M': M})
    
    if intact and d0:
        max_diff = max(abs(a - b) for a, b in zip(intact['marginals'], d0['marginals']))
        print(f'    D0 vs intact max diff: {max_diff:.8f}')
        if max_diff < 1e-6:
            print('    PASS — identical')
        else:
            print('    FAIL — D0 should be identical to intact')

elapsed = time.time() - t0
secs_per_target = elapsed / 2  # 2 targets (intact + D0 for each doc)
print(f'\n  Smoke test: {elapsed:.0f}s total, {secs_per_target:.0f}s per target')
print(f'  Est. pilot run: ~{secs_per_target * 86 * 3 / 3600:.1f} hrs (86 targets × 3 conditions)')

In [ ]:
# === Main loop: intact + D3 + D4 on selected corpora ===

CONDITIONS = {
    'intact': {'fn': lambda ctx: ctx, 'kwargs': {}},
    'D3_M50': {'fn': d3_swap_halves, 'kwargs': {'M': 50}},
    'D4': {'fn': d4_full_reverse, 'kwargs': {}},
}

def run_conditions_on_corpus(corpus_name, corpus_path, probe_name):
    docs = []
    with open(corpus_path) as f:
        for line in f:
            docs.append(json.loads(line))
    
    for cond_name, cond_spec in CONDITIONS.items():
        cache_path = BASE / f'{probe_name}_{corpus_name}_{cond_name}.json'
        if cache_path.exists():
            with open(cache_path) as f:
                cached = json.load(f)
            print(f'  {cond_name}: cached ({len(cached)} results)')
            continue
        
        print(f'  {cond_name}: processing {len(docs)} docs...')
        t0 = time.time()
        results = []
        
        for doc in tqdm(docs, desc=f'{corpus_name}/{cond_name}'):
            full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
            n = len(full_ids)
            
            for frac in TARGET_FRACS:
                target_start = int(n * frac)
                target_end = min(target_start + TARGET_LEN, n)
                if target_start < MIN_BEFORE or target_end - target_start < 5:
                    continue
                
                r = process_one_target(
                    full_ids, target_start, target_end,
                    cond_spec['fn'],
                    cond_spec['kwargs'] if cond_spec['kwargs'] else None
                )
                if r:
                    r['doc_id'] = doc.get('doc_id', '')
                    r['target_frac'] = frac
                    results.append(r)
        
        elapsed = time.time() - t0
        with open(cache_path, 'w') as f:
            json.dump(results, f)
        print(f'    {len(results)} results in {elapsed/60:.1f} min')
        
        if results:
            mean_marg = np.mean([np.mean(r['marginals']) for r in results])
            print(f'    Mean raw marginal: {mean_marg:.4f}')

for probe_name in PROBE_LIST:
    print(f'\n{"="*60}')
    print(f'{probe_name.upper()}: intact + D3 + D4 on {list(CORPORA.keys())}')
    print(f'{"="*60}')
    
    if probe_name == 'llama' and 'model' in dir() and model is not None:
        print('Llama already loaded')
    else:
        if 'model' in dir() and model is not None:
            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()
        tokenizer = AutoTokenizer.from_pretrained(PROBES[probe_name])
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            PROBES[probe_name],
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16),
            device_map='auto'
        )
        model.eval()
        print(f'{probe_name} loaded')
    
    for corpus_name, corpus_path in CORPORA.items():
        print(f'\n--- {corpus_name} ---')
        if not corpus_path.exists():
            print(f'  NOT FOUND')
            continue
        run_conditions_on_corpus(corpus_name, corpus_path, probe_name)
    
    # === Per-probe summary ===
    print(f'\n{"="*60}')
    print(f'{probe_name.upper()} SUMMARY')
    print(f'{"="*60}')
    print(f'{"Corpus":<15} {"Cond":<10} {"Mean marg":>10} {"Detail":>35}')
    print('-' * 73)
    
    for corpus_name in CORPORA:
        for cond in ['intact', 'D3_M50', 'D4']:
            cp = BASE / f'{probe_name}_{corpus_name}_{cond}.json'
            if not cp.exists(): continue
            with open(cp) as f: results = json.load(f)
            if not results: continue
            
            all_marg = np.array([r['marginals'] for r in results])
            mean_curve = np.mean(all_marg, axis=0)
            mean_val = np.mean(mean_curve)
            
            detail = ''
            if cond == 'D3_M50' and len(mean_curve) > M:
                jump = mean_curve[M] - mean_curve[M-1]
                pre_sd = np.std(mean_curve[:M])
                z = jump / pre_sd if pre_sd > 0 else 0
                detail = f'jump at M+1={jump:.4f} (z={z:.1f})'
            elif cond == 'D4':
                ip = BASE / f'{probe_name}_{corpus_name}_intact.json'
                if ip.exists():
                    with open(ip) as f: ir = json.load(f)
                    intact_curve = np.mean([r['marginals'] for r in ir], axis=0)
                    rev_intact = intact_curve[::-1]
                    rho_rev, _ = stats.spearmanr(mean_curve, rev_intact)
                    rho_fwd, _ = stats.spearmanr(mean_curve, intact_curve)
                    detail = f'corr(D4,rev_intact)={rho_rev:.3f} vs fwd={rho_fwd:.3f}'
            
            print(f'{corpus_name:<15} {cond:<10} {mean_val:>10.4f} {detail:>35}')

print('\nDone!')

In [ ]:
# === Visualization: raw marginal curves ===
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

n_corpora = len(CORPORA)
ncols = min(4, n_corpora)
nrows = (n_corpora + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
axes_flat = axes.flatten()

for idx, corpus_name in enumerate(CORPORA):
    ax = axes_flat[idx]
    
    for cond, color in [('intact', 'blue'), ('D3_M50', 'red'), ('D4', 'green')]:
        for probe in PROBE_LIST:
            cp = BASE / f'{probe}_{corpus_name}_{cond}.json'
            if not cp.exists(): continue
            with open(cp) as f: results = json.load(f)
            if not results: continue
            mean_curve = np.mean([r['marginals'] for r in results], axis=0)
            smooth = uniform_filter1d(mean_curve, 5)
            ax.plot(range(1, len(smooth)+1), smooth, color=color,
                    linewidth=2, label=cond, alpha=0.9)
            break
    
    ax.axvline(M, color='gray', linestyle=':', alpha=0.5, label=f'M={M}')
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(corpus_name, fontweight='bold')
    ax.set_xlabel('Distance (tokens)')
    ax.set_ylabel('Raw Marginal (PPL drop)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

for idx in range(n_corpora, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('D3 (swap halves) and D4 (full reverse) vs Intact — Raw Marginals',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D3_D4_raw.png', dpi=150, bbox_inches='tight')
plt.show()

# Also plot the PPL curves directly (not marginals) — easier to see the reversal
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
axes_flat = axes.flatten()

for idx, corpus_name in enumerate(CORPORA):
    ax = axes_flat[idx]
    
    for cond, color in [('intact', 'blue'), ('D3_M50', 'red'), ('D4', 'green')]:
        for probe in PROBE_LIST:
            cp = BASE / f'{probe}_{corpus_name}_{cond}.json'
            if not cp.exists(): continue
            with open(cp) as f: results = json.load(f)
            if not results: continue
            # PPL curves (c=0 to C)
            mean_ppl = np.mean([r['ppls'] for r in results], axis=0)
            ax.plot(range(0, len(mean_ppl)), mean_ppl, color=color,
                    linewidth=2, label=cond, alpha=0.9)
            break
    
    ax.axvline(M, color='gray', linestyle=':', alpha=0.5, label=f'M={M}')
    ax.set_title(corpus_name, fontweight='bold')
    ax.set_xlabel('Context length (tokens)')
    ax.set_ylabel('Perplexity')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

for idx in range(n_corpora, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('PPL Curves: How perplexity changes with context length',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D3_D4_ppl_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figures saved')